In [0]:
# Databricks notebook source
# Validation

from datetime import datetime, timezone
from decimal import Decimal

BRONZE = "workspace.default.bronze_ventas"
SILVER = "workspace.default.silver_ventas"
GOLD = "workspace.default.gold_ventas"
RESULTADOS = "workspace.default.validacion_resultados"

# COMMAND ----------

# Una sola pasada por la silver para casi todas las reglas
s = spark.sql(f"""
SELECT COUNT(DISTINCT date) AS dias,
       CAST(MIN(date) AS STRING) AS primera,
       CAST(MAX(date) AS STRING) AS ultima,
       COUNT(DISTINCT store, date, check_number) AS ordenes,
       COUNT_IF(pop IN ('To Go', 'Inside', '3')) AS pop_viejos,
       COUNT_IF(tender_type IN ('$1', '$5', '$10', '$20', '$50')) AS billetes_sueltos,
       COUNT(DISTINCT pop) AS valores_pop,
       COUNT(DISTINCT tender_type) AS valores_tender,
       COUNT_IF(store IS NULL OR pos_code IS NULL OR check_number IS NULL
                OR menu_price IS NULL OR total IS NULL OR item_discount IS NULL
                OR total_discount IS NULL OR tax IS NULL OR item_voided IS NULL
                OR order_voided IS NULL OR transaction_type IS NULL OR pop IS NULL
                OR date IS NULL OR begin_time IS NULL OR end_time IS NULL
                OR duracion_min IS NULL) AS filas_con_nulos,
       CAST(SUM(menu_price) AS STRING) AS suma_precios,
       SUM(duracion_min) AS suma_minutos
FROM {SILVER}
""").first()

# Por orden: total es de la orden, así que debe ser el mismo en todos sus renglones
o = spark.sql(f"""
SELECT COUNT_IF(total_min <> total_max) AS ordenes_con_dos_totales,
       SUM(total_max) AS venta_por_orden
FROM (
  SELECT MIN(total) AS total_min, MAX(total) AS total_max
  FROM {SILVER}
  GROUP BY store, date, check_number
)
""").first()

c = spark.sql(f"""
SELECT (SELECT COUNT(*) FROM {BRONZE}) AS filas_bronze,
       (SELECT COUNT(*) FROM {SILVER}) AS filas_silver,
       (SELECT COUNT(*) FROM {GOLD})   AS filas_gold
""").first()

diferencia_venta = abs(float(o.venta_por_orden) - float(Decimal(s.suma_precios))) / float(Decimal(s.suma_precios))

# COMMAND ----------

# (dimensión, regla, valor, esperado, severidad)
checks = [
    ("completitud",  "días con datos",                          s.dias,                    573,              "fail"),
    ("completitud",  "primer día",                              s.primera,                 "2025-01-22",     "fail"),
    ("completitud",  "último día",                              s.ultima,                  "2026-09-27",     "fail"),
    ("unicidad",     "órdenes (store + date + check_number)",   s.ordenes,                 13771296,         "fail"),
    ("unicidad",     "órdenes con más de un total",             o.ordenes_con_dos_totales, 0,                "warn"),
    ("consistencia", "etiquetas viejas en pop",                 s.pop_viejos,              0,                "fail"),
    ("consistencia", "billetes sueltos en tender_type",         s.billetes_sueltos,        0,                "fail"),
    ("consistencia", "valores distintos de pop",                s.valores_pop,             9,                "fail"),
    ("consistencia", "valores distintos de tender_type",        s.valores_tender,          14,               "fail"),
    ("validez",      "filas con nulos en columnas obligatorias", s.filas_con_nulos,        0,                "fail"),
    ("exactitud",    "filas en bronze",                         c.filas_bronze,            85180477,         "fail"),
    ("exactitud",    "filas en silver",                         c.filas_silver,            100330857,        "fail"),
    ("exactitud",    "filas en gold iguales a silver",          c.filas_gold,              c.filas_silver,   "fail"),
    ("exactitud",    "suma de precios",                         s.suma_precios,            "193331740.1900", "fail"),
    ("exactitud",    "suma de minutos",                         s.suma_minutos,            177528042,        "fail"),
    ("exactitud",    "venta por orden vs suma de precios (≤ 2 %)", diferencia_venta <= 0.02, True,          "warn"),
]

corrida = datetime.now(timezone.utc)
filas = [(corrida, d, r, str(v), str(e), sev, v == e) for d, r, v, e, sev in checks]

resultados = spark.createDataFrame(
    filas,
    "corrida TIMESTAMP, dimension STRING, regla STRING, valor STRING, esperado STRING, severidad STRING, ok BOOLEAN",
)
resultados.write.mode("append").saveAsTable(RESULTADOS)
display(resultados)

fallas = [f"{d}: {r} (salió {v}, se esperaba {e})" for _, d, r, v, e, sev, ok in filas if sev == "fail" and not ok]
if fallas:
    raise Exception("La validación no pasó:\n" + "\n".join(fallas))
print(f"Validación OK: {len(filas)} reglas revisadas; diferencia venta vs precios = {diferencia_venta:.2%}")